In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("aocdml.keras")
last_conv_layer_name = model.get_layer("conv2d_3")

# # To check for last conv layer name
# for layer in model.layers:
#     print(layer.name)

In [ ]:
import numpy as np

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


In [ ]:
import shap

def make_shap_explanation(img_array, model):
    background_data = np.load("background_data.npy")
    explainer = shap.GradientExplainer(model, background_data)
    shap_values = explainer.shap_values(img_array)
    return shap_values


In [ ]:
from PIL import Image
import cv2
import matplotlib.pyplot as plt

# Loading Image
img = Image.open("test_image.jpg")
img = img.resize((224, 224))  # resize
img_array = np.array(img) / 255.0  # normalize
img_array = np.expand_dims(img_array, axis=0)

# Making Prediction and Classifying Image
prediction = model.predict(img_array)
predicted_class = np.argmax(prediction, axis=1)

# Overlaying Heatmap on OG Image
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
heatmap = np.uint8(255 * heatmap)
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
img_display = np.uint8(255 * img)
superimposed_img = cv2.addWeighted(img_display, 0.6, heatmap, 0.4, 0)
# superimposed_img = cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB) # If it looks weird

# Collecting SHAP values
shap_values = make_shap_explanation(img_array, model)

# Returning Outputs
print("Predicted class:", predicted_class[0])
print("Probabilities:", prediction)

plt.imshow(superimposed_img)
plt.axis("off")
plt.title("Grad-CAM")
plt.show()

shap.image_plot([shap_values[predicted_class]], img_array)